# moonshot_ml_sandbox — V1 dataset builder

**Purpose.** Build a self-contained, leakage-safe `(security, date) -> feature-vector / target-vector`
dataset from the frozen `moonshot_compounding` v3 feature panel, as a learning
sandbox for vector math -> linear regression -> neural networks.

**This is a learning project, not a research finding.** It is *not* logged as a
hypothesis test and carries no DSR / hold-out machinery. Leakage discipline is
retained in full anyway — because a leak would make the *learning* fake, not
because it would make a *strategy* fake.

**What this notebook does, in order:**
1. Mount Drive + preflight the mount.
2. Load the newest v3 panel **read-only**.
3. Filter to the point-in-time Nifty 500 universe (`in_nifty500_flag == 1`).
4. Compute five backward-looking features (through close of day *t*).
5. Compute one forward target (open *t+1* -> open *t+21*), across a one-day leakage wall.
6. Drop rows with any NaN feature / NaN target; drop dates with < 30 names. Every drop logged.
7. Cross-sectionally z-score each feature **within each surviving date**.
8. Print six validation blocks — coverage funnel, leakage proof, spot-checks,
   normalization check, per-date universe check, window-thinness report.
9. **Stop and wait for review.** Only the final cell writes files.

**Spec (settled):**

| Item | Decision |
|---|---|
| Universe | `in_nifty500_flag == 1` (explicit `== 1`, tri-state safe) |
| Features (5) | momentum 12-1, reversal (1m), realized_vol_60, log-turnover, overnight-drift (21d) |
| Normalization | cross-sectional z-score within each date, applied **after** all filtering |
| Target | absolute forward return, `open[t+1] -> open[t+21]` |
| Leakage wall | features <= close(t); target starts open(t+1); 1-day gap |
| Vendor-failure date | 2025-03-18 excluded from any compounding window (not zero-filled) |
| Min cross-section | drop dates with < 30 eligible names |
| Window-thinness | positional count, reported not filtered (V1 lean) |
| Residualization | NOT in V1 (parked as V2 equal-weighted / V3 Nifty 500) |

**Non-negotiables carried from the main project:** read-only on the panel;
writes only into `moonshot_ml_sandbox/`; timestamped outputs, never overwritten;
NaN means unknown (never interpolated / zero-filled); no silent drops.

## Cell 0 — Mount Google Drive + preflight

A stale Colab Drive mount presents as an empty/again-syncing folder: the
directory tree appears but file contents don't. That is **not** lost data. If
the preflight below sees the folder but zero panel files, the fix is
*Runtime > Disconnect and delete runtime*, then re-run this cell.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')  # prompts for auth on first run; no-op if already mounted

FEATURES_DIR = Path("/content/drive/MyDrive/moonshot_compounding/features")

assert FEATURES_DIR.exists(), (
    f"{FEATURES_DIR} does not exist. Either the path is wrong or the mount is "
    "stale. If you can see the folder in the Drive UI but this fails, do: "
    "Runtime > Disconnect and delete runtime, then re-run this cell."
)

panels = sorted(FEATURES_DIR.glob("feature_panel_with_pit_v2_*.parquet"))
print(f"Mount OK. {FEATURES_DIR} is visible.")
print(f"Found {len(panels)} feature-panel file(s).")
if panels:
    print(f"Newest: {panels[-1].name}")
else:
    print("WARNING: folder visible but no feature-panel files found — classic "
          "stale-mount signature. Disconnect-and-delete the runtime and remount.")

Mounted at /content/drive
Mount OK. /content/drive/MyDrive/moonshot_compounding/features is visible.
Found 4 feature-panel file(s).
Newest: feature_panel_with_pit_v2_2026-07-24.parquet


## Cell 1 — Imports, config, and the column map

Everything name-dependent keys off `COLMAP`. If the panel schema ever changes,
this is the single place to fix it. The config block also holds every tunable
number in the spec, so the parameters of the build are visible in one spot
rather than scattered through the code.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date

# --- Paths -------------------------------------------------------------------
COMPOUNDING_FEATURES = Path("/content/drive/MyDrive/moonshot_compounding/features")
SANDBOX_DIR          = Path("/content/drive/MyDrive/moonshot_ml_sandbox")

# --- Column map: the ONE place to fix a name if the panel schema changes -----
# Left = role this notebook needs; right = actual column in the v3 panel
# (verified against the read-only probe run 2026-07-24).
COLMAP = {
    "ticker":      "ticker",
    "date":        "Date",
    "open":        "Open",
    "close":       "Close",
    "volume":      "Volume",
    "turnover":    "turnover",
    "daily_ret":   "daily_return",
    "overnight":   "overnight_return",
    "rvol60":      "realized_vol_60",
    "flag":        "in_nifty500_flag",
    "date_status": "date_status",
    "source":      "source",
}
C = COLMAP  # shorthand

# --- Spec parameters (every tunable number, in one place) --------------------
MOM_START      = 252   # momentum window begins t-252
MOM_END        = 21    # momentum window ends t-21 (skip last month)
REV_WINDOW     = 21    # reversal = last 21 trading rows
OVN_WINDOW     = 21    # overnight-drift = trailing mean of 21 overnight returns
TARGET_ENTRY   = 1     # target buys at open of t+1 (the leakage wall)
TARGET_HOLD    = 20    # ...and sells 20 trading sessions later => open[t+21]
MIN_XSEC       = 30    # drop any date with fewer than this many eligible names
VENDOR_FAILURE_DATES = {pd.Timestamp("2025-03-18")}  # fact #24/#25 — exclude from windows

# A running log of every row/date we drop and why. Printed as a funnel later.
BUILD_LOG = []
def log_step(label, n_rows, n_tickers=None, n_dates=None, note=""):
    entry = {"step": label, "rows": int(n_rows),
             "tickers": None if n_tickers is None else int(n_tickers),
             "dates":   None if n_dates   is None else int(n_dates),
             "note": note}
    BUILD_LOG.append(entry)
    t = "" if n_tickers is None else f" | {n_tickers:>5} tickers"
    d = "" if n_dates   is None else f" | {n_dates:>5} dates"
    print(f"{label:<46} {n_rows:>12,} rows{t}{d}  {note}")

print("Config loaded. Parameters:")
print(f"  momentum window : t-{MOM_START} .. t-{MOM_END}")
print(f"  reversal window : last {REV_WINDOW} rows")
print(f"  overnight drift : trailing {OVN_WINDOW} rows")
print(f"  target          : open[t+{TARGET_ENTRY}] -> open[t+{TARGET_ENTRY+TARGET_HOLD}]")
print(f"  min cross-sec   : {MIN_XSEC} names/date")
print(f"  vendor-failure  : {sorted(d.date().isoformat() for d in VENDOR_FAILURE_DATES)}")

Config loaded. Parameters:
  momentum window : t-252 .. t-21
  reversal window : last 21 rows
  overnight drift : trailing 21 rows
  target          : open[t+1] -> open[t+21]
  min cross-sec   : 30 names/date
  vendor-failure  : ['2025-03-18']


## Cell 2 — Load the frozen panel (read-only) and restrict to the PIT universe

We read the newest `feature_panel_with_pit_v2_*.parquet` and immediately keep
only rows genuinely in the index that day. **The filter is `== 1`, never `!= 0`**:
the flag is tri-state (1 / 0 / NaN), and `!= 1` would silently bundle the
pre-2010-06-30 NaN "unknown" flags in with the known-outs (fact #17).

We keep the whole panel around as `panel_full` too, because the feature windows
for an in-universe row on date *t* may reach back into dates when the same
ticker was *not* in the universe. The universe filter decides **which rows
become cards**, not **which history the features may see**. Those are different
questions and conflating them would corrupt the momentum window at every point a
stock recently entered the index.

In [3]:
panels = sorted(COMPOUNDING_FEATURES.glob("feature_panel_with_pit_v2_*.parquet"))
assert panels, f"No feature panel found in {COMPOUNDING_FEATURES} (stale mount?)."
PANEL_PATH = panels[-1]
print(f"Reading (newest-wins, READ-ONLY): {PANEL_PATH.name}")

panel_full = pd.read_parquet(PANEL_PATH)
panel_full[C["date"]] = pd.to_datetime(panel_full[C["date"]]).dt.tz_localize(None)
panel_full = panel_full.sort_values([C["ticker"], C["date"]]).reset_index(drop=True)

log_step("0. raw panel loaded",
         len(panel_full),
         panel_full[C["ticker"]].nunique(),
         panel_full[C["date"]].nunique())

# The set of (ticker, date) rows that are ELIGIBLE TO BECOME CARDS.
# Explicit == 1 (tri-state safe). This is the universe filter.
eligible = panel_full[panel_full[C["flag"]] == 1].copy()
log_step("1. flag == 1 (PIT universe -> cards)",
         len(eligible),
         eligible[C["ticker"]].nunique(),
         eligible[C["date"]].nunique())

Reading (newest-wins, READ-ONLY): feature_panel_with_pit_v2_2026-07-24.parquet
0. raw panel loaded                               3,328,132 rows |  1069 tickers |  4237 dates  
1. flag == 1 (PIT universe -> cards)              1,973,146 rows |  1069 tickers |  4059 dates  


## Cell 3 — Per-ticker feature & target engine

The heart of the build. For each ticker we walk its own price history (all of
it, in-universe or not) and, for every row that is an eligible card, compute the
five features looking **backward** and the one target looking **forward**.

**Why an explicit per-ticker loop and not a clever vectorized one-liner?** Two
reasons. First, the windows must be counted in *trading rows*, skipping the
vendor-failure date — that bookkeeping is easy to get subtly wrong in a
vectorized rolling call and nearly impossible to eyeball afterwards. Second,
this is a learning notebook: the loop is meant to be *read*. It is slower; on a
frozen panel we run it once, so slowness is free. Correctness we can see beats
speed we have to trust.

**The leakage wall, made physical.** For a card on row `i` (date *t*):
- features use rows with index **< i's window ends** — never row `i`'s own
  future, never anything at or after the target entry;
- the target uses `open` at row `i + TARGET_ENTRY` (t+1) and row
  `i + TARGET_ENTRY + TARGET_HOLD` (t+21).
The gap between "last row a feature may touch" and "first row the target
touches" is the wall. The code records the exact dates used so Cell 5 can prove
the wall held.

**Compounding with holes.** A window's return is `prod(1 + r) - 1` over the
daily returns actually present. Missing days (ordinary no-trades *and* the
vendor-failure date) are **dropped from the product, never treated as zero** — a
zero is a fabricated claim that the stock didn't move. We also count how many
real returns went into each window (the positional thinness report, V1-lean:
measured, not filtered).

In [4]:
def compound_returns(daily_returns):
    # Compound a sequence of simple daily returns: prod(1+r) - 1.
    # NaNs are DROPPED (not zero-filled). Returns (value, n_used).
    r = daily_returns[~np.isnan(daily_returns)]
    n_used = int(r.size)
    if n_used == 0:
        return np.nan, 0
    return float(np.prod(1.0 + r) - 1.0), n_used


def build_cards_for_ticker(g):
    # g: one ticker's full history, date-sorted. Returns a list of card dicts.
    # Uses positional row indexing within g, which is how the windows are defined.
    g = g.reset_index(drop=True)
    n = len(g)

    dates      = g[C["date"]].to_numpy()
    opens      = g[C["open"]].to_numpy()
    dret       = g[C["daily_ret"]].to_numpy()
    overnight  = g[C["overnight"]].to_numpy()
    rvol60     = g[C["rvol60"]].to_numpy()
    turnover   = g[C["turnover"]].to_numpy()
    flag       = g[C["flag"]].to_numpy()
    dstatus    = g[C["date_status"]].to_numpy()

    # Boolean mask of rows that are the vendor-failure date — excluded from
    # every compounding window (fact #25).
    is_vendor = np.array([pd.Timestamp(d) in VENDOR_FAILURE_DATES for d in dates])

    cards = []
    for i in range(n):
        # This row becomes a CARD only if it is in the PIT universe.
        if flag[i] != 1:
            continue

        # ---- Window index bounds (positional, within this ticker) ----------
        mom_lo, mom_hi = i - MOM_START, i - MOM_END      # [t-252, t-21)
        rev_lo, rev_hi = i - REV_WINDOW, i                # [t-21,  t)
        ovn_lo, ovn_hi = i - OVN_WINDOW, i                # [t-21,  t)
        tgt_entry = i + TARGET_ENTRY                      # t+1
        tgt_exit  = i + TARGET_ENTRY + TARGET_HOLD        # t+21

        # Need history for the longest backward window and future for the target.
        if mom_lo < 0 or tgt_exit >= n:
            continue

        # ---- Vendor-failure exclusion inside each window --------------------
        # Build index arrays, then drop any that land on the vendor-failure date.
        def keep(idx_lo, idx_hi):
            idx = np.arange(idx_lo, idx_hi)
            return idx[~is_vendor[idx]]

        mom_idx = keep(mom_lo, mom_hi)
        rev_idx = keep(rev_lo, rev_hi)
        ovn_idx = keep(ovn_lo, ovn_hi)

        # ---- Features (all backward, all <= close at end of their window) ---
        mom_val, mom_n = compound_returns(dret[mom_idx])
        rev_val, rev_n = compound_returns(dret[rev_idx])

        # Overnight drift: trailing mean of overnight returns (NaNs dropped).
        ovn_slice = overnight[ovn_idx]
        ovn_slice = ovn_slice[~np.isnan(ovn_slice)]
        ovn_val   = float(np.mean(ovn_slice)) if ovn_slice.size else np.nan
        ovn_n     = int(ovn_slice.size)

        # Realized vol: read straight from the panel at row i (already t-aligned).
        rv_val = rvol60[i]

        # Log-turnover at row i. Guard non-positive turnover -> NaN (log undefined).
        tv = turnover[i]
        logturn = float(np.log(tv)) if (tv is not None and tv > 0) else np.nan

        # ---- Target: open[t+1] -> open[t+21], vendor day cannot be an endpoint
        # If either endpoint is the vendor-failure date, the target is undefined
        # for this card (we do not silently shift to a different day).
        if is_vendor[tgt_entry] or is_vendor[tgt_exit]:
            tgt_val = np.nan
        else:
            o_in, o_out = opens[tgt_entry], opens[tgt_exit]
            tgt_val = float(o_out / o_in - 1.0) if (o_in and o_in > 0
                        and not np.isnan(o_in) and not np.isnan(o_out)) else np.nan

        cards.append({
            "ticker": g[C["ticker"]].iloc[i],
            "date":   pd.Timestamp(dates[i]),
            # features
            "momentum_12_1":  mom_val,
            "reversal_1m":    rev_val,
            "realized_vol_60": rv_val,
            "log_turnover":   logturn,
            "overnight_drift": ovn_val,
            # target
            "fwd_ret_20d":    tgt_val,
            # thinness bookkeeping (positional; measured, not filtered)
            "mom_n_used":     mom_n,
            "mom_n_full":     int(MOM_START - MOM_END),   # 231 slots nominal
            "rev_n_used":     rev_n,
            "rev_n_full":     int(REV_WINDOW),
            # leakage-proof breadcrumbs (dates the wall sits between)
            "_feat_last_date": pd.Timestamp(dates[i]),                 # close(t)
            "_tgt_entry_date": pd.Timestamp(dates[tgt_entry]),         # open(t+1)
            "_tgt_exit_date":  pd.Timestamp(dates[tgt_exit]),          # open(t+21)
        })
    return cards


# Run the engine over every ticker. Uses panel_full (full history), not the
# pre-filtered eligible frame, because features may look back into out-of-
# universe history — the flag check inside decides which rows become cards.
all_cards = []
tickers = panel_full[C["ticker"]].unique()
print(f"Building cards for {len(tickers):,} tickers... (single pass, ~a few min)")
for k, (tkr, g) in enumerate(panel_full.groupby(C["ticker"], sort=False)):
    all_cards.extend(build_cards_for_ticker(g))
    if (k + 1) % 200 == 0:
        print(f"  {k+1:>5}/{len(tickers)} tickers processed, "
              f"{len(all_cards):,} cards so far")

cards = pd.DataFrame(all_cards)
log_step("2. cards built (features + target computed)",
         len(cards),
         cards["ticker"].nunique(),
         cards["date"].nunique(),
         note="pre-NaN-filter")

Building cards for 1,069 tickers... (single pass, ~a few min)
    200/1069 tickers processed, 376,856 cards so far
    400/1069 tickers processed, 745,495 cards so far
    600/1069 tickers processed, 1,096,300 cards so far
    800/1069 tickers processed, 1,420,608 cards so far
   1000/1069 tickers processed, 1,750,925 cards so far
2. cards built (features + target computed)       1,881,417 rows |  1041 tickers |  3905 dates  pre-NaN-filter


## Cell 4 — Drop NaNs and thin dates (each drop logged), then z-score

Now the honest attrition. In order, logging every step so nothing vanishes
silently:

1. Drop cards with **any NaN feature** (warm-up edges, empty windows).
2. Drop cards with a **NaN target** (the handful of missing-open rows, plus any
   target endpoint on the vendor day).
3. Drop **dates with fewer than 30 surviving names** — a z-score across a handful
   of names is noise wearing a number's clothes.
4. **Then** z-score each feature cross-sectionally *within each date*, over
   exactly the rows that survived — so the mean/std reference the final set,
   not rows we were about to drop.

In [5]:
FEATURES = ["momentum_12_1", "reversal_1m", "realized_vol_60",
            "log_turnover", "overnight_drift"]
TARGET = "fwd_ret_20d"

# 1. any NaN feature
before = len(cards)
cards = cards.dropna(subset=FEATURES)
log_step("3. drop cards with any NaN feature", len(cards),
         cards["ticker"].nunique(), cards["date"].nunique(),
         note=f"-{before-len(cards):,}")

# 2. NaN target
before = len(cards)
cards = cards.dropna(subset=[TARGET])
log_step("4. drop cards with NaN target", len(cards),
         cards["ticker"].nunique(), cards["date"].nunique(),
         note=f"-{before-len(cards):,}")

# 3. thin dates (< MIN_XSEC names)
before = len(cards)
date_counts = cards.groupby("date")["ticker"].transform("size")
dropped_dates = sorted(cards.loc[date_counts < MIN_XSEC, "date"].unique())
cards = cards[date_counts >= MIN_XSEC].copy()
log_step(f"5. drop dates with < {MIN_XSEC} names", len(cards),
         cards["ticker"].nunique(), cards["date"].nunique(),
         note=f"-{before-len(cards):,} rows, {len(dropped_dates)} dates")

# 4. cross-sectional z-score within each date, over the SURVIVING rows.
# Assign via groupby index to avoid the apply-drops-grouping-col trap.
def zscore_within_date(s):
    mu, sd = s.mean(), s.std(ddof=0)
    return (s - mu) / sd if sd > 0 else s * 0.0  # sd==0 -> all identical -> 0

for f in FEATURES:
    cards[f + "_z"] = cards.groupby("date")[f].transform(zscore_within_date)

print("\\nZ-scored feature columns added:", [f + "_z" for f in FEATURES])

3. drop cards with any NaN feature                1,815,739 rows |  1041 tickers |  3825 dates  -65,678
4. drop cards with NaN target                     1,813,757 rows |  1041 tickers |  3823 dates  -1,982
5. drop dates with < 30 names                     1,813,407 rows |  1041 tickers |  3756 dates  -350 rows, 67 dates
\nZ-scored feature columns added: ['momentum_12_1_z', 'reversal_1m_z', 'realized_vol_60_z', 'log_turnover_z', 'overnight_drift_z']


## Cell 5 — Validation (shown, not claimed)

Six blocks, printed before anything is written. This is the whole ethic: the
notebook doesn't ask you to believe it, it shows you.

In [6]:
print("=" * 78)
print("VALIDATION BLOCK 1 — COVERAGE FUNNEL")
print("=" * 78)
funnel = pd.DataFrame(BUILD_LOG)
print(funnel.to_string(index=False))
print(f"\\nFinal dataset: {len(cards):,} cards | "
      f"{cards['ticker'].nunique():,} tickers | {cards['date'].nunique():,} dates")
print(f"Date range: {cards['date'].min().date()} -> {cards['date'].max().date()}")

VALIDATION BLOCK 1 — COVERAGE FUNNEL
                                       step    rows  tickers  dates                note
                        0. raw panel loaded 3328132     1069   4237                    
       1. flag == 1 (PIT universe -> cards) 1973146     1069   4059                    
2. cards built (features + target computed) 1881417     1041   3905      pre-NaN-filter
         3. drop cards with any NaN feature 1815739     1041   3825             -65,678
              4. drop cards with NaN target 1813757     1041   3823              -1,982
              5. drop dates with < 30 names 1813407     1041   3756 -350 rows, 67 dates
\nFinal dataset: 1,813,407 cards | 1,041 tickers | 3,756 dates
Date range: 2011-01-03 -> 2026-06-10


In [7]:
print("=" * 78)
print("VALIDATION BLOCK 2 — LEAKAGE PROOF (one sample card)")
print("=" * 78)
# Pick a sample card well inside the panel (not an edge) for a clean illustration.
sample = cards.iloc[len(cards) // 2]
print(f"Sample card: {sample['ticker']} on {sample['date'].date()}")
print(f"  feature information cutoff (close of t) : {sample['_feat_last_date'].date()}")
print(f"  target ENTRY  (open of t+1)             : {sample['_tgt_entry_date'].date()}")
print(f"  target EXIT   (open of t+21)            : {sample['_tgt_exit_date'].date()}")
gap_ok = sample['_feat_last_date'] < sample['_tgt_entry_date']
print(f"\\n  WALL HOLDS?  feature cutoff ({sample['_feat_last_date'].date()}) "
      f"< target entry ({sample['_tgt_entry_date'].date()}) : {gap_ok}")
assert gap_ok, "LEAKAGE: feature cutoff is not strictly before target entry!"
print("  -> features use only information available at/before close(t);")
print("     target is realised strictly after, from open(t+1). No overlap.")

VALIDATION BLOCK 2 — LEAKAGE PROOF (one sample card)
Sample card: JKIL.NS on 2018-05-17
  feature information cutoff (close of t) : 2018-05-17
  target ENTRY  (open of t+1)             : 2018-05-18
  target EXIT   (open of t+21)            : 2018-06-15
\n  WALL HOLDS?  feature cutoff (2018-05-17) < target entry (2018-05-18) : True
  -> features use only information available at/before close(t);
     target is realised strictly after, from open(t+1). No overlap.


In [8]:
print("=" * 78)
print("VALIDATION BLOCK 3 — MANUAL SPOT-CHECKS (recompute from raw prices)")
print("=" * 78)
# Independently recompute momentum and the target for 2 sample cards straight
# from the panel's raw Close/Open, and compare to the builder's stored values.
def recompute_one(tkr, tdate):
    g = panel_full[panel_full[C["ticker"]] == tkr].sort_values(C["date"]).reset_index(drop=True)
    i = g.index[g[C["date"]] == tdate][0]
    # momentum: compound daily_return over [i-252, i-21), dropping vendor day
    idx = np.arange(i - MOM_START, i - MOM_END)
    dates_w = g[C["date"]].to_numpy()[idx]
    idx = idx[~np.array([pd.Timestamp(d) in VENDOR_FAILURE_DATES for d in dates_w])]
    r = g[C["daily_ret"]].to_numpy()[idx]
    r = r[~np.isnan(r)]
    mom = np.prod(1 + r) - 1
    # target: open[i+1] -> open[i+21]
    o_in  = g[C["open"]].iloc[i + TARGET_ENTRY]
    o_out = g[C["open"]].iloc[i + TARGET_ENTRY + TARGET_HOLD]
    tgt = o_out / o_in - 1
    return mom, tgt

for _, row in cards.sample(2, random_state=0).iterrows():
    mom_re, tgt_re = recompute_one(row["ticker"], row["date"])
    print(f"\\n{row['ticker']} on {row['date'].date()}")
    print(f"  momentum : builder={row['momentum_12_1']:+.6f}  "
          f"recomputed={mom_re:+.6f}  diff={abs(row['momentum_12_1']-mom_re):.2e}")
    print(f"  target   : builder={row['fwd_ret_20d']:+.6f}  "
          f"recomputed={tgt_re:+.6f}  diff={abs(row['fwd_ret_20d']-tgt_re):.2e}")

VALIDATION BLOCK 3 — MANUAL SPOT-CHECKS (recompute from raw prices)
\nPFIZER.NS on 2016-03-02
  momentum : builder=+0.132757  recomputed=+0.132757  diff=0.00e+00
  target   : builder=-0.069302  recomputed=-0.069302  diff=0.00e+00
\nPRAJIND.NS on 2024-02-06
  momentum : builder=+0.616332  recomputed=+0.616332  diff=0.00e+00
  target   : builder=-0.001645  recomputed=-0.001645  diff=0.00e+00


In [9]:
print("=" * 78)
print("VALIDATION BLOCK 4 — NORMALIZATION CHECK (one sample date)")
print("=" * 78)
# On a sample date, each z-scored feature should have mean ~0 and std ~1.
sdate = cards["date"].value_counts().index[len(cards["date"].unique()) // 2]
day = cards[cards["date"] == sdate]
print(f"Sample date: {sdate.date()}  ({len(day)} names)")
print(f"{'feature':<22}{'mean(z)':>12}{'std(z)':>12}")
for f in FEATURES:
    z = day[f + "_z"]
    print(f"{f:<22}{z.mean():>12.2e}{z.std(ddof=0):>12.4f}")
print("  -> means ~0, stds ~1 confirm z-scoring is within-date and correct.")

VALIDATION BLOCK 4 — NORMALIZATION CHECK (one sample date)
Sample date: 2011-07-14  (488 names)
feature                    mean(z)      std(z)
momentum_12_1            -1.27e-17      1.0000
reversal_1m               9.10e-19      1.0000
realized_vol_60          -2.20e-16      1.0000
log_turnover              3.95e-16      1.0000
overnight_drift           9.46e-17      1.0000
  -> means ~0, stds ~1 confirm z-scoring is within-date and correct.


In [10]:
print("=" * 78)
print("VALIDATION BLOCK 5 — PER-DATE UNIVERSE (min-cross-section filter bit)")
print("=" * 78)
per_date = cards.groupby("date")["ticker"].size()
print(f"names per date -> min {per_date.min()}, "
      f"median {int(per_date.median())}, max {per_date.max()}")
print(f"dates with < {MIN_XSEC} names remaining: {int((per_date < MIN_XSEC).sum())} "
      f"(must be 0)")
assert (per_date >= MIN_XSEC).all(), "A sub-threshold date survived the filter!"
print("  -> confirmed: every surviving date has >= 30 names.")

VALIDATION BLOCK 5 — PER-DATE UNIVERSE (min-cross-section filter bit)
names per date -> min 31, median 488, max 501
dates with < 30 names remaining: 0 (must be 0)
  -> confirmed: every surviving date has >= 30 names.


In [11]:
print("=" * 78)
print("VALIDATION BLOCK 6 — WINDOW-THINNESS REPORT (positional; measured, not filtered)")
print("=" * 78)
# How many cards were built on a FULL window vs a thin one? V1 does NOT filter on
# this — it reports it, so you can decide with data whether V2 needs a
# minimum-observations guard.
for name, used, full in [("momentum_12_1", "mom_n_used", "mom_n_full"),
                         ("reversal_1m",   "rev_n_used", "rev_n_full")]:
    frac = cards[used] / cards[full]
    print(f"\\n{name}  (nominal full window = {int(cards[full].iloc[0])} returns)")
    print(f"  full (100%)        : {(frac >= 0.999).mean()*100:6.2f}% of cards")
    print(f"  90-100%            : {((frac >= 0.90) & (frac < 0.999)).mean()*100:6.2f}%")
    print(f"  70-90%             : {((frac >= 0.70) & (frac < 0.90)).mean()*100:6.2f}%")
    print(f"  < 70%              : {(frac < 0.70).mean()*100:6.2f}%")
    print(f"  thinnest window    : {int(cards[used].min())} returns "
          f"({frac.min()*100:.1f}% of full)")
print("\\n  -> V1 keeps all of these. Use this distribution to decide whether V2 "
      "should impose a minimum-observations rule.")

VALIDATION BLOCK 6 — WINDOW-THINNESS REPORT (positional; measured, not filtered)
\nmomentum_12_1  (nominal full window = 231 returns)
  full (100%)        :  91.95% of cards
  90-100%            :   7.97%
  70-90%             :   0.04%
  < 70%              :   0.04%
  thinnest window    : 38 returns (16.5% of full)
\nreversal_1m  (nominal full window = 21 returns)
  full (100%)        : 100.00% of cards
  90-100%            :   0.00%
  70-90%             :   0.00%
  < 70%              :   0.00%
  thinnest window    : 21 returns (100.0% of full)
\n  -> V1 keeps all of these. Use this distribution to decide whether V2 should impose a minimum-observations rule.


## Cell 6 — Write outputs (the ONLY cell that writes anything)

Run this **only after** reviewing the six validation blocks above. Writes two
timestamped files into `moonshot_ml_sandbox/`, never overwriting:
- the dataset (cards) as parquet,
- the coverage funnel as a CSV audit trail.

The internal `_`-prefixed breadcrumb date columns are dropped from the saved
dataset (they were only there to prove the wall); everything else is kept.

In [12]:
SANDBOX_DIR.mkdir(parents=True, exist_ok=True)
today = date.today().isoformat()

# Drop the leakage-proof breadcrumbs; keep raw + z-scored features, target, thinness.
drop_cols = [c for c in cards.columns if c.startswith("_")]
out = cards.drop(columns=drop_cols)

dataset_path = SANDBOX_DIR / f"ml_sandbox_v1_dataset_{today}.parquet"
funnel_path  = SANDBOX_DIR / f"ml_sandbox_v1_buildlog_{today}.csv"

# Refuse same-day overwrite (files are timestamped and never overwritten).
for p in (dataset_path, funnel_path):
    assert not p.exists(), f"{p.name} already exists — refusing to overwrite. " \
                           "Delete it first if you intend to rebuild."

out.to_parquet(dataset_path, index=False)
pd.DataFrame(BUILD_LOG).to_csv(funnel_path, index=False)

print(f"WROTE dataset : {dataset_path}")
print(f"      shape   : {out.shape[0]:,} rows x {out.shape[1]} cols")
print(f"      columns : {out.columns.tolist()}")
print(f"WROTE funnel  : {funnel_path}")
print("\\nV1 sandbox dataset built. Next: visualize the feature space before any modelling.")

WROTE dataset : /content/drive/MyDrive/moonshot_ml_sandbox/ml_sandbox_v1_dataset_2026-09-07.parquet
      shape   : 1,813,407 rows x 17 cols
      columns : ['ticker', 'date', 'momentum_12_1', 'reversal_1m', 'realized_vol_60', 'log_turnover', 'overnight_drift', 'fwd_ret_20d', 'mom_n_used', 'mom_n_full', 'rev_n_used', 'rev_n_full', 'momentum_12_1_z', 'reversal_1m_z', 'realized_vol_60_z', 'log_turnover_z', 'overnight_drift_z']
WROTE funnel  : /content/drive/MyDrive/moonshot_ml_sandbox/ml_sandbox_v1_buildlog_2026-09-07.csv
\nV1 sandbox dataset built. Next: visualize the feature space before any modelling.
